In [17]:
import pandas as pd
import random
import os

What we want to achieve: to have a dataframe called stimuli.csv that we can import into lab.js. It will have a primes, a targets, and a conditions column.
Why do we need this? Because it's much easier than typing it all in by hand into the lab.js interface

How are we going to get this dataframe?\
    1. import the original dataframe\
    2. find stimuli pairs for each condition and create a new dataframe for each condition with the columns 'prime' and 'target'\
    3. create a new column that has the prime + target word pairs: this is 'combined'\
    4. create sets of unique word pairs for each condition from the 'combined' column\
    5. convert sets to lists and shuffle them\
    6. make each shuffled list the same length\
    7. re-split the pairs so they can be in their own columns - we need this for the stimuli.csv dataframe\
    8. putting the dataset together:
> 1. columns with all the primes, all the targets, and condition labels\
> 2. create a datafram of these columns using pandas\
> 3. save datafram to a csv to import it to lab.js


1. import the original dataframe

In [18]:
# we want to define where we want to get the original datafile from, 
# and where we're going to put the file that we created
# we also want to give the names of the files

# paths and file names
pathin = "/work/ExPsyLing_E2026/LDT"
pathout = "/work/ExPsyLing_E2026/LDT/Second_try"
infile = "Original_data_LDT.csv"
outfile = "New_stimuli.csv"

os.makedirs(pathout, exist_ok=True)  # creates the output folder if it doesn't exist

In [19]:
raw = pd.read_csv(os.path.join(pathin, infile), sep=";", low_memory=False)


# read in the data
# raw = 
# we don't want warnings, bc this is a big file, so put low_memory = False at the end

2. find stimuli pairs for each condition and create a new dataframe for each condition with the columns 'prime' and 'target'

lexicality 1 = words and 2 = nonwords, split almost exactly in half.
primecond 1 to 4 matches one type/rel combination each: 1 = related first associate, 2 = related other, 3 = unrelated first, 4 = unrelated other.
All nonword rows are nw/nw, whatever their primecond. The single #NULL! row is a data glitch, and it doesn't matter because we select on lexicality.

In [20]:
# create a new dataframe for the three conditions: related, unrelated, and nonword
# step 1: what are the names of these columns in the dataframe?
# step 2: what goes in the last bracket for each new dataframe?

# df_condition1 = raw[(raw['C1'] == 'A') & raw(['C2'] == 'B')][['X', 'Y']]
# df_condition_nw = raw[raw['C1'] == 'C'][['X', 'Y']]

df_related   = raw[raw['rel'] == 'rel'][['prime', 'target']].copy()
df_unrelated = raw[raw['rel'] == 'un'][['prime', 'target']].copy()
df_nonword   = raw[raw['rel'] == 'nw'][['prime', 'target']].copy()

print(df_related.shape, df_unrelated.shape, df_nonword.shape)

cond_dfs = {
    "related": df_related,
    "unrelated": df_unrelated,
    "nonword": df_nonword,
}

(211959, 2) (211928, 2) (423836, 2)


In [21]:
# 3. combine prime and target into one string per row
# "|" is the separator so we can split the pairs again in step 7
for name, d in cond_dfs.items():
    d["combined"] = d["prime"].astype(str).str.strip() + "|" + d["target"].astype(str).str.strip()

# 4. unique word pairs per condition, as sets
pair_sets = {}
for name, d in cond_dfs.items():
    pair_sets[name] = set(d["combined"])
    print(name, "unique pairs:", len(pair_sets[name]))

related unique pairs: 3323
unrelated unique pairs: 3323
nonword unique pairs: 26116


In [25]:
# 5. convert sets to lists and shuffle them
# sorted() first: set order can change between runs, so sorting makes
# the shuffle reproducible together with random.seed(42)
raw = pd.read_csv(os.path.join(pathin, infile), sep=";", low_memory=False, keep_default_na=False)

pair_lists = {}
for name, s in pair_sets.items():
    lst = sorted(s)
    random.shuffle(lst)
    pair_lists[name] = lst

# 6. make every list the same length (trim to the shortest)
n = min(len(lst) for lst in pair_lists.values())
print("common length:", n)

for name in pair_lists:
    pair_lists[name] = pair_lists[name][:n]
    print(name, len(pair_lists[name]))

TypeError: '<' not supported between instances of 'float' and 'str'

In [26]:
# re-read without turning strings like "nan"/"null" into missing values
raw = pd.read_csv(os.path.join(pathin, infile), sep=";", low_memory=False,
                  keep_default_na=False)

# 2. one dataframe per condition
cond_dfs = {
    "related":   raw[raw["rel"] == "rel"][["prime", "target"]].copy(),
    "unrelated": raw[raw["rel"] == "un"][["prime", "target"]].copy(),
    "nonword":   raw[raw["rel"] == "nw"][["prime", "target"]].copy(),
}

# 3 + 4. combined column and unique pairs (empty cells are dropped)
pair_sets = {}
for name, d in cond_dfs.items():
    d["prime"] = d["prime"].astype(str).str.strip()
    d["target"] = d["target"].astype(str).str.strip()
    empty = (d["prime"] == "") | (d["target"] == "")
    print(name, "rows with empty prime/target:", empty.sum())
    d = d[~empty].copy()
    d["combined"] = d["prime"] + "|" + d["target"]
    cond_dfs[name] = d
    pair_sets[name] = set(d["combined"])
    print(name, "unique pairs:", len(pair_sets[name]))

# 5. sets -> sorted lists -> shuffled
random.seed(42)
pair_lists = {}
for name, s in pair_sets.items():
    lst = sorted(s)
    random.shuffle(lst)
    pair_lists[name] = lst

# 6. trim to the shortest length
n = min(len(lst) for lst in pair_lists.values())
print("common length:", n)
for name in pair_lists:
    pair_lists[name] = pair_lists[name][:n]
    print(name, len(pair_lists[name]))

related rows with empty prime/target: 0
related unique pairs: 3323
unrelated rows with empty prime/target: 0
unrelated unique pairs: 3323


nonword rows with empty prime/target: 0
nonword unique pairs: 26131
common length: 3323
related 3323
unrelated 3323
nonword 3323


3. create a new column that has the prime + target word pairs: this is 'combined'

In [ ]:
# create a new column for each dataframe that combines the prime and target into a single string, 
# so we can easily identify unique pairs

# df_condition1['combined'] = df_condition1['X'] + " " + df_condition1['Y']

4. create sets of unique word pairs for each condition from the 'combined' column

In [ ]:
# create sets of unique pairs for each condition
# why do we need sets?
# what type of collection variable are our pairs stored in now?
# what do we have to do to make them into sets?

# condition1 = 

5. convert sets to lists and shuffle them

In [ ]:
# convert the sets to lists so we can shuffle them


In [ ]:
# shuffle the lists so that we can randomly select stimuli from them
# random.shuffle(condition1_list)

6. make each shuffled list the same length

In [ ]:
# select the first 20 stimuli from each condition
# step 1: how do we select from 0 to 20?

7. re-split the pairs so they can be in their own columns - we need this for the stimuli.csv dataframe

In [ ]:
# now that we have picked stimulus pairs, re-split them so they can be in each their column

# condition1_targets = [stim.split(" ")[1] for stim in condition1_stimuli]
# condition1_primes = [stim.split(" ")[0] for stim in condition1_stimuli]

8. putting the dataset together:
        1. columns with all the primes, all the taergets, and condition labels

In [ ]:
# make a list with all the primes, and a list with all the targets
# primes = 
# targets = 

In [ ]:
# make a list with the condition labels for each stimulus pair
# how many labels do we want? 
# conditions = ['condition1'] * howmanylabels? + ['condition2'] * howmanylabels? + ['condition3'] * howmanylabels?


2. create a dataframe of these columns using pandas

In [ ]:
# create a new dataframe with the selected stimuli and their conditions
# step 1: how do we make a pandas dataframe?
# step 2: use ({}) when creating the dataframe. You want the columns you made eariler and their assigned names inside {}
# df_stimuli = 

3. save datafram to a csv to import it to lab.js

In [ ]:
# save the new dataframe to a CSV file, for importing into labjs
# df_stimuli.to_csv(pathout + outfile, index = False)